# Experiment 3 — Standalone: weight=4 extended check (does the sweet spot hold up?)

Self-contained, no dependency on any prior kernel state. Sections 0-6 are the grammar/model/
training/eval code verbatim from the original notebook. Section 7 is the big-table diagnostic
machinery. Section 8 is the reweighted-loss fix with the override-fired split metric. Section 9
is the actual check we want: does `override_weight=4` (the best trade-off point found in the
weight sweep: fired_acc~0.80, not_fired_acc~0.91 at 20k steps, still rising) hold up or improve
further with double the training (40k steps, 2 seeds)?

Run top to bottom.

## 0. Setup

In [1]:
import os
GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import random, time, math, pickle
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(0)
    print('device: cuda ->', torch.cuda.get_device_name(0))
else:
    print('device: cpu (no GPU visible -- check GPU_ID / CUDA_VISIBLE_DEVICES if unexpected)')


device: cuda -> NVIDIA A100 80GB PCIe


## 1. Shared PCFG structure

In [2]:
import random
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

N_COUNT_NOUNS = 300
N_MASS_NOUNS  = 300
COUNT_NOUNS = [f"cnt{i:04d}" for i in range(N_COUNT_NOUNS)]
MASS_NOUNS  = [f"mss{i:04d}" for i in range(N_MASS_NOUNS)]
NOUN_CLASS = {n: 'count' for n in COUNT_NOUNS}
NOUN_CLASS.update({n: 'mass' for n in MASS_NOUNS})
ALL_NOUNS = COUNT_NOUNS + MASS_NOUNS

QUANTIFIERS = ['q_sev', 'q_man']

N_VTR = 6
N_VINTR = 4
VERB_STEMS_TR = [f"vtr{i}" for i in range(N_VTR)]
VERB_STEMS_INTR = [f"vin{i}" for i in range(N_VINTR)]
PREPS = ['p0', 'p1', 'p2', 'p3']
RPRO = 'rp'
CONJ = 'cj'
DET = 'dt'

P_S_COORD        = 0.20
P_MOD_PRESENT    = 0.45
P_MOD_IS_REL     = 0.50
P_TRANSITIVE     = 0.55
P_OBJ_MOD        = 0.30
P_QUANT_PRESENT  = 0.35
P_ATTRACTOR_PL   = 0.5


def sample_number(rng: random.Random) -> str:
    # FIX: was `random.choice(...)` (global stream, ignored the seeded rng passed
    # everywhere else in the grammar) -- made every "same seed" comparison across
    # runs/sessions non-reproducible, since this decides the lexical number of
    # every noun (subject AND attractor). Now threaded through like every other
    # draw in the grammar.
    return rng.choice(['SG', 'PL'])


@dataclass
class NP:
    noun: str
    noun_class: str
    quant: Optional[str]
    lexical_number: str
    mod: Optional[dict]
    final_number: str = None


def sample_np(rng: random.Random, allow_mod=True) -> NP:
    noun = rng.choice(ALL_NOUNS)
    ncls = NOUN_CLASS[noun]
    lex_num = sample_number(rng)  # FIX: was sample_number()
    quant = rng.choice(QUANTIFIERS) if rng.random() < P_QUANT_PRESENT else None
    mod = None
    if allow_mod and rng.random() < P_MOD_PRESENT:
        attractor_noun = rng.choice(ALL_NOUNS)
        mod = dict(
            kind='PP' if rng.random() < P_MOD_IS_REL else 'REL',
            attractor_noun=attractor_noun,
            attractor_number=sample_number(rng),  # FIX: was sample_number()
        )
    return NP(noun=noun, noun_class=ncls, quant=quant, lexical_number=lex_num, mod=mod)


@dataclass
class Clause:
    subj: NP
    verb_stem: str
    transitive: bool
    obj: Optional[NP]


def sample_clause(rng: random.Random) -> Clause:
    subj = sample_np(rng, allow_mod=True)
    transitive = rng.random() < P_TRANSITIVE
    if transitive:
        obj = sample_np(rng, allow_mod=(rng.random() < P_OBJ_MOD))
        verb_stem = rng.choice(VERB_STEMS_TR)
    else:
        obj = None
        verb_stem = rng.choice(VERB_STEMS_INTR)
    return Clause(subj=subj, verb_stem=verb_stem, transitive=transitive, obj=obj)


@dataclass
class Sentence:
    clauses: List[Clause]
    coordinated: bool


def sample_sentence(rng: random.Random) -> Sentence:
    c1 = sample_clause(rng)
    if rng.random() < P_S_COORD:
        c2 = sample_clause(rng)
        return Sentence(clauses=[c1, c2], coordinated=True)
    return Sentence(clauses=[c1], coordinated=False)


## 2. Per-language parameters: τ and ε

In [3]:
from dataclasses import dataclass
from typing import Dict, Tuple
import random

OVERRIDE_CELLS = [(q, c) for q in QUANTIFIERS for c in ('count', 'mass')]
ANCHOR_OVERRIDE: Dict[Tuple[str, str], str] = {
    ('q_sev', 'count'): 'PL',
    ('q_sev', 'mass'):  'NONE',
    ('q_man', 'count'): 'NONE',
    ('q_man', 'mass'):  'SG',
}


def build_override_table(eps_flips: int, seed: int) -> Dict[Tuple[str, str], str]:
    rng = random.Random(seed)
    table = dict(ANCHOR_OVERRIDE)
    cells_ = list(OVERRIDE_CELLS)
    rng.shuffle(cells_)
    outcomes = ['NONE', 'SG', 'PL']
    for cell in cells_[:eps_flips]:
        current = table[cell]
        table[cell] = rng.choice([o for o in outcomes if o != current])
    return table


@dataclass
class LangSpec:
    name: str
    tau_intervene: float
    eps_flips: int
    seed: int
    anchor: bool = False

    def __post_init__(self):
        self.override_table = (dict(ANCHOR_OVERRIDE) if self.anchor
                                else build_override_table(self.eps_flips, self.seed))

    def apply_override(self, np_: NP) -> str:
        if np_.quant is None:
            return np_.lexical_number
        forced = self.override_table[(np_.quant, np_.noun_class)]
        return np_.lexical_number if forced == 'NONE' else forced

    def mod_in_situ(self, rng: random.Random) -> bool:
        return rng.random() < self.tau_intervene


## 3. Surface realization (includes override_fired annotation)

In [4]:
import random
from typing import List, Dict, Tuple

BOS, EOS = '<BOS>', '<EOS>'
DOT = '.'
DET_SG, DET_PL = 'dt_sg', 'dt_pl'

VFORMS = {(stem, n): f"agr_{stem}_{n}" for stem in VERB_STEMS_TR + VERB_STEMS_INTR
          for n in ('SG', 'PL')}


def realize_np(np_: NP, lang: LangSpec, role: str) -> Tuple[List[str], str]:
    toks = []
    if np_.quant:
        toks.append(np_.quant)
    det = DET_PL if np_.lexical_number == 'PL' else DET_SG
    toks += [det, np_.noun]
    final_number = lang.apply_override(np_)
    return toks, final_number


def realize_mod(np_: NP, rng: random.Random) -> List[str]:
    mod = np_.mod
    if mod is None:
        return []
    det = DET_PL if mod['attractor_number'] == 'PL' else DET_SG
    if mod['kind'] == 'PP':
        prep = rng.choice(['p0', 'p1', 'p2', 'p3'])
        return [prep, det, mod['attractor_noun']]
    else:
        return ['rp', det, mod['attractor_noun']]


def realize_clause(clause: Clause, lang: LangSpec, rng: random.Random, ann: List[dict], toks: List[str]):
    """Includes override_fired: whether the quantifier override actually changed the
    answer away from the noun's own marked default at this agreement position --
    needed so the loss can be reweighted toward exactly the rare cases the model
    tends to ignore."""
    subj_toks, subj_final_num = realize_np(clause.subj, lang, role='subj')
    subj_mod_toks = realize_mod(clause.subj, rng)
    mod_in_situ = lang.mod_in_situ(rng) if clause.subj.mod is not None else False

    toks += subj_toks
    if clause.subj.mod is not None and mod_in_situ:
        toks += subj_mod_toks

    verb_pos = len(toks)
    vform = VFORMS[(clause.verb_stem, subj_final_num)]
    toks.append(vform)
    override_fired = (clause.subj.quant is not None and subj_final_num != clause.subj.lexical_number)
    ann.append(dict(pos=verb_pos, subcomp='subj_verb_agree',
                     target_number=subj_final_num,
                     had_intervening_attractor=bool(clause.subj.mod is not None and mod_in_situ),
                     attractor_mismatches=(clause.subj.mod is not None and mod_in_situ and
                                           clause.subj.mod['attractor_number'] != subj_final_num),
                     override_fired=override_fired))

    if clause.subj.mod is not None and not mod_in_situ:
        toks += subj_mod_toks

    if clause.transitive and clause.obj is not None:
        obj_toks, _ = realize_np(clause.obj, lang, role='obj')
        obj_mod_toks = realize_mod(clause.obj, rng)
        toks += obj_toks
        if clause.obj.mod is not None:
            toks += obj_mod_toks


def realize_sentence(sent: Sentence, lang: LangSpec, rng: random.Random) -> Tuple[List[str], List[dict]]:
    toks = [BOS]
    ann = []
    for i, clause in enumerate(sent.clauses):
        realize_clause(clause, lang, rng, ann, toks)
        if i == 0 and sent.coordinated:
            toks.append('cj')
    toks.append(DOT)
    toks.append(EOS)
    return toks, ann


## 4. Tokenizer / shared vocabulary

In [5]:
import random
import torch
from typing import List

SPECIAL = ['<PAD>', BOS, EOS]
VOCAB = SPECIAL + [DOT, DET_SG, DET_PL, 'rp', 'cj'] + PREPS + QUANTIFIERS + ALL_NOUNS + list(VFORMS.values())
VOCAB = list(dict.fromkeys(VOCAB))
STOI = {t: i for i, t in enumerate(VOCAB)}
ITOS = {i: t for t, i in STOI.items()}
PAD_ID = STOI['<PAD>']
VOCAB_SIZE = len(VOCAB)

VFORM_SG_IDS = {stem: STOI[VFORMS[(stem, 'SG')]] for stem in VERB_STEMS_TR + VERB_STEMS_INTR}
VFORM_PL_IDS = {stem: STOI[VFORMS[(stem, 'PL')]] for stem in VERB_STEMS_TR + VERB_STEMS_INTR}


def make_example(lang: LangSpec, rng: random.Random, max_len: int):
    s = sample_sentence(rng)
    toks, ann = realize_sentence(s, lang, rng)
    if len(toks) > max_len:
        return None
    ids = [STOI[t] for t in toks]
    return ids, ann, toks


def make_batch(lang: LangSpec, rng: random.Random, batch_size: int, max_len: int = 24):
    seqs, anns = [], []
    while len(seqs) < batch_size:
        ex = make_example(lang, rng, max_len)
        if ex is None:
            continue
        ids, ann, toks = ex
        seqs.append(ids)
        anns.append(ann)
    x = torch.full((batch_size, max_len), PAD_ID, dtype=torch.long)
    for i, s in enumerate(seqs):
        x[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    return x, anns


## 5. Model: capacity-bottlenecked TinyGPT

In [6]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_len):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        mask = torch.tril(torch.ones(max_len, max_len)).view(1, 1, max_len, max_len)
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(hd)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_embd_tok, n_embd=64, n_layer=3, n_head=4, max_len=24, pad_id=0):
        super().__init__()
        self.tok_bottleneck = nn.Embedding(vocab_size, n_embd_tok, padding_idx=pad_id)
        self.tok_proj = nn.Linear(n_embd_tok, n_embd) if n_embd_tok != n_embd else nn.Identity()
        self.pos_emb = nn.Embedding(max_len, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.max_len = max_len

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_proj(self.tok_bottleneck(idx)) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        return self.head(x)


## 6. Baseline training + feature-level evaluation

In [7]:
import random
import torch
import torch.nn.functional as F

def compute_loss(model, x, device):
    x = x.to(device)
    logits = model(x[:, :-1])
    targets = x[:, 1:].contiguous()
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                            ignore_index=PAD_ID)
    return loss


def train_isolated(lang: LangSpec, n_embd_tok, steps=1500, seed=0, lr=2e-2,
                    n_embd=64, n_layer=3, n_head=4, max_len=24, device='cpu'):
    torch.manual_seed(seed)
    model = TinyGPT(VOCAB_SIZE, n_embd_tok, n_embd, n_layer, n_head, max_len, PAD_ID).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr * 0.02)
    rng = random.Random(1000 + seed)
    for step in range(steps):
        opt.zero_grad()
        x, _ = make_batch(lang, rng, batch_size=64, max_len=max_len)
        loss = compute_loss(model, x, device)
        loss.backward()
        opt.step()
        sched.step()
    return model


@torch.no_grad()
def feature_accuracy(model, lang: LangSpec, device='cpu', n_eval=4000, max_len=24, seed=999):
    model.eval()
    rng = random.Random(seed)
    correct = {'all': 0}
    total = {'all': 0}
    n_done = 0
    while n_done < n_eval:
        bs = min(64, n_eval - n_done)
        x, anns = make_batch(lang, rng, batch_size=bs, max_len=max_len)
        x = x.to(device)
        logits = model(x[:, :-1])
        for i, ann_list in enumerate(anns):
            for a in ann_list:
                pos = a['pos']
                if pos == 0 or pos >= x.shape[1]:
                    continue
                stem_token_id = x[i, pos].item()
                stem = None
                for s, sg_id in VFORM_SG_IDS.items():
                    if sg_id == stem_token_id or VFORM_PL_IDS[s] == stem_token_id:
                        stem = s
                        break
                sg_id, pl_id = VFORM_SG_IDS[stem], VFORM_PL_IDS[stem]
                l_sg = logits[i, pos - 1, sg_id].item()
                l_pl = logits[i, pos - 1, pl_id].item()
                pred = 'SG' if l_sg > l_pl else 'PL'
                is_correct = int(pred == a['target_number'])
                correct['all'] += is_correct
                total['all'] += 1
        n_done += bs
    return {k: correct[k] / total[k] for k in total}


## 7. Leakage / calibration audit

In [8]:
import random

def audit_rates(lang: LangSpec, n=20000, seed=0):
    rng = random.Random(seed)
    n_agree = n_intervene = n_mismatch = n_quant = n_override_active = 0
    for _ in range(n):
        s = sample_sentence(rng)
        toks, ann = realize_sentence(s, lang, rng)
        for a in ann:
            n_agree += 1
            n_intervene += int(a['had_intervening_attractor'])
            n_mismatch += int(a['attractor_mismatches'])
        for cl in s.clauses:
            if cl.subj.quant is not None:
                n_quant += 1
                if lang.apply_override(cl.subj) != cl.subj.lexical_number:
                    n_override_active += 1
    return dict(
        intervene_rate=n_intervene / n_agree,
        recency_would_be_wrong_rate=n_mismatch / n_agree,
        quant_present_rate=n_quant / n_agree,
        override_changes_answer_rate=(n_override_active / n_quant) if n_quant else 0.0,
    )


## 8. Big table + reweighted loss with split metric

In [9]:
import random as _random

def build_big_table_globals(n_quant: int, n_class: int, anchor_seed: int = 12345):
    global QUANTIFIERS, NOUN_CLASS, OVERRIDE_CELLS, ANCHOR_OVERRIDE
    global VOCAB, STOI, ITOS, VOCAB_SIZE, VFORM_SG_IDS, VFORM_PL_IDS, PAD_ID

    QUANTIFIERS = [f"q{i}" for i in range(n_quant)]
    classes = [f"cls{i}" for i in range(n_class)]
    NOUN_CLASS = {}
    for idx, noun in enumerate(ALL_NOUNS):
        NOUN_CLASS[noun] = classes[idx % n_class]

    OVERRIDE_CELLS = [(q, c) for q in QUANTIFIERS for c in classes]

    rng = _random.Random(anchor_seed)
    outcomes_weighted = ['NONE', 'NONE', 'SG', 'PL']
    ANCHOR_OVERRIDE = {cell: rng.choice(outcomes_weighted) for cell in OVERRIDE_CELLS}

    VOCAB = SPECIAL + [DOT, DET_SG, DET_PL, 'rp', 'cj'] + PREPS + QUANTIFIERS + ALL_NOUNS + list(VFORMS.values())
    VOCAB = list(dict.fromkeys(VOCAB))
    STOI = {t: i for i, t in enumerate(VOCAB)}
    ITOS = {i: t for t, i in STOI.items()}
    PAD_ID = STOI['<PAD>']
    VOCAB_SIZE = len(VOCAB)
    VFORM_SG_IDS = {stem: STOI[VFORMS[(stem, 'SG')]] for stem in VERB_STEMS_TR + VERB_STEMS_INTR}
    VFORM_PL_IDS = {stem: STOI[VFORMS[(stem, 'PL')]] for stem in VERB_STEMS_TR + VERB_STEMS_INTR}

    print(f"rebuilt globals: n_quant={n_quant} n_class={n_class} "
          f"table_size={len(OVERRIDE_CELLS)} vocab_size={VOCAB_SIZE}")


def compute_loss_weighted(model, x, anns, device, override_weight):
    x = x.to(device)
    logits = model(x[:, :-1])
    targets = x[:, 1:].contiguous()
    B, Tm1 = targets.shape
    per_tok = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                               ignore_index=PAD_ID, reduction='none').view(B, Tm1)
    weight = torch.ones(B, Tm1, device=device)
    for i, ann_list in enumerate(anns):
        for a in ann_list:
            pos = a['pos']
            if pos == 0 or pos >= x.shape[1]:
                continue
            if a.get('override_fired', False):
                weight[i, pos - 1] = override_weight
    pad_mask = (targets != PAD_ID).float()
    weight = weight * pad_mask
    return (per_tok * weight).sum() / weight.sum().clamp_min(1.0)


@torch.no_grad()
def feature_accuracy_split(model, lang, device='cpu', n_eval=4000, max_len=24, seed=999):
    model.eval()
    rng = random.Random(seed)
    correct = {'fired': 0, 'not_fired': 0}
    total = {'fired': 0, 'not_fired': 0}
    n_done = 0
    while n_done < n_eval:
        bs = min(64, n_eval - n_done)
        x, anns = make_batch(lang, rng, batch_size=bs, max_len=max_len)
        x = x.to(device)
        logits = model(x[:, :-1])
        for i, ann_list in enumerate(anns):
            for a in ann_list:
                pos = a['pos']
                if pos == 0 or pos >= x.shape[1]:
                    continue
                stem_token_id = x[i, pos].item()
                stem = None
                for s, sg_id in VFORM_SG_IDS.items():
                    if sg_id == stem_token_id or VFORM_PL_IDS[s] == stem_token_id:
                        stem = s
                        break
                sg_id, pl_id = VFORM_SG_IDS[stem], VFORM_PL_IDS[stem]
                l_sg = logits[i, pos - 1, sg_id].item()
                l_pl = logits[i, pos - 1, pl_id].item()
                pred = 'SG' if l_sg > l_pl else 'PL'
                is_correct = int(pred == a['target_number'])
                key = 'fired' if a.get('override_fired', False) else 'not_fired'
                correct[key] += is_correct
                total[key] += 1
        n_done += bs
    out = {}
    for k in total:
        out[k] = correct[k] / total[k] if total[k] else float('nan')
    out['n_fired'] = total['fired']
    out['n_not_fired'] = total['not_fired']
    return out


def train_isolated_weighted_split(lang, n_embd_tok, steps, override_weight, seed=0, lr=2e-2,
                                   n_embd=64, n_layer=3, n_head=4, max_len=24, device='cpu',
                                   eval_every=2000, n_eval=4000):
    torch.manual_seed(seed)
    model = TinyGPT(VOCAB_SIZE, n_embd_tok, n_embd, n_layer, n_head, max_len, PAD_ID).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr * 0.02)
    rng = random.Random(1000 + seed)
    running_loss = 0.0
    log = []
    t0 = time.time()
    for step in range(1, steps + 1):
        opt.zero_grad()
        x, anns = make_batch(lang, rng, batch_size=64, max_len=max_len)
        loss = compute_loss_weighted(model, x, anns, device, override_weight)
        loss.backward()
        opt.step()
        sched.step()
        running_loss += loss.item()
        if step % eval_every == 0:
            avg_loss = running_loss / eval_every
            split = feature_accuracy_split(model, lang, device=device, n_eval=n_eval)
            log.append(dict(step=step, loss=avg_loss, **split, t=time.time() - t0))
            print(f"[{time.time()-t0:6.0f}s] step={step:6d}  loss={avg_loss:.4f}  "
                  f"fired_acc={split['fired']:.4f} (n={split['n_fired']})  "
                  f"not_fired_acc={split['not_fired']:.4f} (n={split['n_not_fired']})")
            running_loss = 0.0
            model.train()
    return model, log


## 9. The actual check: does weight=4 hold up (or improve) at 40,000 steps?

From the weight sweep (20k steps): weight=4 gave fired_acc~0.80, not_fired_acc~0.91, with
fired_acc still visibly climbing at the cutoff (one seed went 0.72->0.79 in just the last
4k steps). Doubling the budget checks whether that climb continues without further cost to
not_fired_acc -- which is what we'd want before treating weight=4 as the setting to build
the real joint bilingual eps sweep on.

In [10]:
build_big_table_globals(n_quant=16, n_class=16, anchor_seed=12345)
A_final = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
rates = audit_rates(A_final, n=8000)
print(f"audit: {rates}")

FINAL_STEPS = 40000
FINAL_WEIGHT = 4.0
FINAL_SEEDS = (0, 1)
final_logs = {}
for seed in FINAL_SEEDS:
    print(f"--- seed {seed} ---")
    model, log = train_isolated_weighted_split(A_final, n_embd_tok=96, steps=FINAL_STEPS,
                                                override_weight=FINAL_WEIGHT, seed=seed,
                                                device=DEVICE, eval_every=2000, n_eval=4000)
    final_logs[seed] = log

with open('exp3_weight4_extended.pkl', 'wb') as f:
    pickle.dump(final_logs, f)
print("\nsaved exp3_weight4_extended.pkl")


rebuilt globals: n_quant=16 n_class=16 table_size=256 vocab_size=648
audit: {'intervene_rate': 0.028651802458845594, 'recency_would_be_wrong_rate': 0.01260679308189206, 'quant_present_rate': 0.3544488435090644, 'override_changes_answer_rate': 0.21957671957671956}
--- seed 0 ---
[    41s] step=  2000  loss=2.8235  fired_acc=0.5522 (n=393)  not_fired_acc=0.8536 (n=4413)
[    80s] step=  4000  loss=2.7710  fired_acc=0.5827 (n=393)  not_fired_acc=0.8520 (n=4413)
[   121s] step=  6000  loss=2.7994  fired_acc=0.4402 (n=393)  not_fired_acc=0.8600 (n=4413)
[   160s] step=  8000  loss=2.7675  fired_acc=0.6031 (n=393)  not_fired_acc=0.7995 (n=4413)
[   199s] step= 10000  loss=2.7960  fired_acc=0.5522 (n=393)  not_fired_acc=0.6465 (n=4413)
[   241s] step= 12000  loss=2.7894  fired_acc=0.5674 (n=393)  not_fired_acc=0.8056 (n=4413)
[   282s] step= 14000  loss=2.7983  fired_acc=0.6031 (n=393)  not_fired_acc=0.7988 (n=4413)
[   322s] step= 16000  loss=2.7826  fired_acc=0.5649 (n=393)  not_fired_acc=0

### How to read it

- **`fired_acc` keeps climbing past ~0.80 toward 0.85-0.90+, `not_fired_acc` holds near 0.90+**:
  weight=4 is the setting — both numbers healthy, this is the clean instrument to build the
  joint bilingual eps sweep on.
- **`fired_acc` plateaus around 0.80 and stays there**: weight=4 may be a genuine ceiling, not
  just "needed more steps" — worth trying weight=5 as an intermediate point between 4 and 6
  before deciding.
- **`not_fired_acc` starts drifting down with more training**: the trade-off isn't fully
  resolved by more steps at this weight, and 20k steps was actually close to the best
  achievable trade-off at weight=4 specifically.